# Support Inbox RAG CLI
LlamaIndex + Ollama + Chroma + PyMuPDF


In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import chromadb
import fitz
from llama_index.core import Document, Settings, StorageContext, VectorStoreIndex
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.vector_stores.chroma import ChromaVectorStore

## Configuration

In [ ]:
BASE_URL = "http://localhost:11434"
LLM_MODEL = "mistral"
EMBED_MODEL = "nomic-embed-text"

DEFAULT_CSV = Path("../data/documents/fake_inbox.csv")
DEFAULT_PDF = Path("../data/pdf/embedding.pdf")
CHROMA_DIR = Path("../data/vector_store/chroma_llamaindex")
COLLECTION_NAME = "support_inbox_llamaindex"

## Load Inbox CSV

In [ ]:
def load_inbox_csv(csv_path: Path) -> list[Document]:
    docs: list[Document] = []
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = (
                f"Support Ticket\n"
                f"Ticket ID: {row.get('ticket_id', '')}\n"
                f"From: {row.get('from_email', '')}\n"
                f"Subject: {row.get('subject', '')}\n"
                f"Priority: {row.get('priority', '')}\n"
                f"Status: {row.get('status', '')}\n"
                f"Message: {row.get('message', '')}"
            )
            docs.append(
                Document(
                    text=text,
                    metadata={
                        "source": str(csv_path),
                        "type": "inbox_ticket",
                        "ticket_id": row.get("ticket_id", ""),
                    },
                )
            )
    return docs

print(f"CSV loader defined. Testing with {DEFAULT_CSV}")
if DEFAULT_CSV.exists():
    csv_docs = load_inbox_csv(DEFAULT_CSV)
    print(f"Loaded {len(csv_docs)} tickets from CSV")
else:
    print(f"CSV file not found: {DEFAULT_CSV}")

## Load PDF with PyMuPDF

In [ ]:
def load_pdf_with_pymupdf(pdf_path: Path) -> list[Document]:
    docs: list[Document] = []
    with fitz.open(pdf_path) as pdf:
        for page_no, page in enumerate(pdf, start=1):
            text = page.get_text("text").strip()
            if not text:
                continue
            docs.append(
                Document(
                    text=text,
                    metadata={
                        "source": str(pdf_path),
                        "type": "wiki_pdf",
                        "page": page_no,
                    },
                )
            )
    return docs

print(f"PDF loader defined. Testing with {DEFAULT_PDF}")
if DEFAULT_PDF.exists():
    pdf_docs = load_pdf_with_pymupdf(DEFAULT_PDF)
    print(f"Loaded {len(pdf_docs)} pages from PDF")
else:
    print(f"PDF file not found: {DEFAULT_PDF}")

## Build or Load Index

In [ ]:
def build_or_load_index(csv_path: Path, pdf_path: Path) -> VectorStoreIndex:
    Settings.llm = Ollama(model=LLM_MODEL, base_url=BASE_URL, request_timeout=120.0)
    Settings.embed_model = OllamaEmbedding(model_name=EMBED_MODEL, base_url=BASE_URL)
    Settings.chunk_size = 900
    Settings.chunk_overlap = 120

    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    chroma_collection = chroma_client.get_or_create_collection(COLLECTION_NAME)

    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    if chroma_collection.count() > 0:
        print(f"Loading existing index from {CHROMA_DIR}")
        return VectorStoreIndex.from_vector_store(vector_store=vector_store)

    print(f"Building new index from CSV and PDF...")
    docs = load_inbox_csv(csv_path) + load_pdf_with_pymupdf(pdf_path)
    return VectorStoreIndex.from_documents(docs, storage_context=storage_context, show_progress=True)

print("Index builder defined")

## Build Query Engine

In [ ]:
def build_query_engine(index: VectorStoreIndex) -> RetrieverQueryEngine:
    retriever = VectorIndexRetriever(index=index, similarity_top_k=4)
    return RetrieverQueryEngine.from_args(retriever=retriever)

print("Query engine builder defined")

## Initialize Index and Query Engine

**Note:** Ollama must be running locally on http://localhost:11434

In [ ]:
# Uncomment to build/load index and query engine
# print("Building/loading index...")
# index = build_or_load_index(DEFAULT_CSV, DEFAULT_PDF)
# query_engine = build_query_engine(index)
# print("Ready!")

## Interactive Q&A

In [ ]:
def run_single_query(query_engine: RetrieverQueryEngine, question: str) -> None:
    """Run a single query and display results with sources"""
    response = query_engine.query(question)
    print("\nAssistant:")
    print(str(response))

    source_nodes = getattr(response, "source_nodes", [])
    if source_nodes:
        print("\nRetrieved sources:")
        for i, node in enumerate(source_nodes, start=1):
            metadata = getattr(node.node, "metadata", {}) or {}
            src = metadata.get("source", "unknown")
            page = metadata.get("page")
            page_info = f" (page {page})" if page else ""
            print(f"- [{i}] {src}{page_info}")

# Example usage (uncomment with initialized query_engine):
# run_single_query(query_engine, "What is the password reset issue?")
# run_single_query(query_engine, "How many support tickets are open?")